# Pydantic格式的使用
## 1、基本使用
举例

In [3]:
import os
from enum import Enum
from typing import Optional

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rprint
load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")

model =init_chat_model(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={
        "thinking": {"type": "disabled"}  # 关键：在这里关闭思考模式
    }
)

In [4]:
from pydantic import BaseModel, Field, ValidationError


class Person(BaseModel):
    """人物信息"""
    name: str =Field(description="姓名")
    age: int =Field(description="年龄")
    occupation: str =Field(description="职业")
structured_model = model.with_structured_output(Person)


result= structured_model .invoke("张三是一名30岁的软件工程师")
print(result)

name='张三' age=30 occupation='软件工程师'


## 2、高级特性
### 2.1、可选字段

In [4]:
from pydantic import BaseModel,Field
from typing import Optional  # 添加这行
class Person(BaseModel):
    """人物信息"""
    name: str =Field(description="姓名")
    age: Optional[int] =Field(description="年龄")
    occupation: str =Field(description="职业")
structured_model = model.with_structured_output(Person)


result= structured_model .invoke("张三是一名软件工程师")
print(result)

name='张三' age=None occupation='软件工程师'


## 2.3情况3：枚举类型

In [5]:
from enum import Enum
class Priority(str, Enum):
    LOW = "低"
    MEDIUM = "中"
    HIGH = "高"

class CustomerInfo(BaseModel):
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")
structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)

name='王小明' phone='138-1234-5678' email=None issue='订单一直没发货' urgency=<Priority.HIGH: '高'>


### 2.4 情况4：列表提取
举例1：

In [6]:
from pydantic import BaseModel,Field
from typing import List
class Person(BaseModel):
    """人物信息"""
    name:str = Field(description="姓名")
    age: int = Field(description="年龄")

class PersonList(BaseModel):
    """人物列表"""
    people:List[Person]

structured_model = model.with_structured_output(PersonList)
result = structured_model.invoke("张三 30岁，李四 40岁")
print(result)




people=[Person(name='张三', age=30), Person(name='李四', age=40)]


In [7]:
class Review(BaseModel):
    """产品评论"""
    product: str
    rating: int = Field(description="评分 1-5")
    pros: List[str] = Field(description="优点列表")
    cons: List[str] = Field(description="缺点列表")
structured_llm = model.with_structured_output(Review)
review = structured_llm.invoke("""
iPhone 17 很棒！摄像头强大，手感好。但是价格贵，没有充电器。4分。
""")
print(review)

product='iPhone 17' rating=4 pros=['摄像头强大', '手感好'] cons=['价格贵', '没有充电器']


In [8]:
class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")
    # 测试
structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""
invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

invoice_number='INV-2024-001' date='2024-01-15' total_amount=1299.0 items=['MacBook Pro', 'AppleCare+']


## 2.6、限制条件
举例：

In [10]:
from pydantic import ValidationError
class User(BaseModel):
    name:str =Field(description="姓名",min_length=2,max_length=10)
    age:int = Field(description="年龄",le=150)
    email:str = Field(description="邮箱")

user1 = User(name="tom",age=20,email="tom@126.com")
print(user1)
try:
    user2 = User(name="tom",age=200,email="wwj123@qq.com")
    print(f"[ok]{user2}")
except ValidationError as e:
    print(f"[error]{e}")

name='tom' age=20 email='tom@126.com'
[error]1 validation error for User
age
  Input should be less than or equal to 150 [type=less_than_equal, input_value=200, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal
